In [ ]:
import os
from pathlib import Path
import numpy as np
np.set_printoptions(precision=3, suppress=True)

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import importlib
import copy
import multiprocessing as mp
import pprint

from datetime import datetime
from pathlib import Path
from IPython.display import HTML
from time import time
from VariablesClass import VariablesClass
from StructureClass import StructureClass
from StateClass import StateClass
from EquilibriumClass import EquilibriumClass
from SupervisorClass import SupervisorClass
from config import CFG
from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed

import plot_funcs, colors, helpers_builders, learning_funcs, file_funcs, numerical_experiments, parallel_buckles_runner

colors_lst, red, custom_cmap = colors.color_scheme()
plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)

In [ ]:
import config
importlib.reload(config)
from config import CFG

import StructureClass
importlib.reload(StructureClass)
from StructureClass import StructureClass

import VariablesClass
importlib.reload(VariablesClass)
from VariablesClass import VariablesClass

import SupervisorClass
importlib.reload(SupervisorClass)
from SupervisorClass import SupervisorClass

importlib.reload(plot_funcs)
importlib.reload(file_funcs)

In [ ]:
# # --- build geometry (all topology stays in StructureClass) ---
# Strctr = StructureClass(CFG, update_scheme=CFG.Train.update_scheme)

# # --- Initiate variables ---
# Variabs = VariablesClass(Strctr, CFG)

# # --- Initiate Supervisor sizes ---
# Sprvsr = SupervisorClass(Strctr, CFG, supress_prints = True)
# Sprvsr.create_dataset(Strctr, CFG, CFG.Train.dataset_sampling)
# tip_pos = np.array([(Strctr.edges-0.01)*Strctr.L, 0])  # 2nd one for PPT presentation Feb2
# tip_angle = -0.02  # 2nd one for PPT presentation Feb2
# Sprvsr.tip_pos_in_t[1] = tip_pos
# Sprvsr.tip_angle_in_t[1] = tip_angle

In [ ]:
init_buckles = list(product([-1, 1], repeat=4))
desired_buckles = copy.copy(init_buckles)

In [ ]:
importlib.reload(plot_funcs)
importlib.reload(file_funcs)
importlib.reload(numerical_experiments)

date = datetime.now().strftime("%Y-%m-%d_%H00")

RUN_DIR = Path(str("parallel_buckles_runs_")+str(date))
RUN_DIR.mkdir(exist_ok=True)

SAVE_GIFS = True
SAVE_PNGS = True
SAVE_CSVS = True
# MAX_WORKERS = max(1, (os.cpu_count() or 2) // 2)
MAX_WORKERS = 8

parallel_module = "parallel_buckles_runner.py"
# Path("parallel_buckles_runner.py").write_text(parallel_module, encoding="utf-8")

from parallel_buckles_runner import run_one_job

jobs = []
for k, desired_buckle_tup in enumerate(desired_buckles):
    for l, init_buckle_tup in enumerate(init_buckles):
#         if k >= l:
#             continue
        if k != 12 and l != 15:
            continue
        if k == l:
            continue
        jobs.append({
            "k": k,
            "l": l,
            "init_buckle_tup": tuple(init_buckle_tup),
            "desired_buckle_tup": tuple(desired_buckle_tup),
            "run_dir": str(RUN_DIR),
            "save_gifs": SAVE_GIFS,
            "save_pngs": SAVE_PNGS,
            "save_csvs": SAVE_CSVS,
        })

loop_length = len(jobs)
final_loss = np.full((len(init_buckles), len(desired_buckles)), np.nan, dtype=float)
results = {}
failed_jobs = []

print(f"Submitting {loop_length} jobs with MAX_WORKERS={MAX_WORKERS}")

ctx = mp.get_context("spawn")
with ProcessPoolExecutor(max_workers=MAX_WORKERS, mp_context=ctx) as ex:
    futures = {ex.submit(run_one_job, job): job for job in jobs}
    for loop_count, fut in enumerate(as_completed(futures), start=1):
        out = fut.result()
        k = out["k"]
        l = out["l"]
        final_loss[k, l] = out["loss"]
        results[(k, l)] = out
        if not out["ok"]:
            failed_jobs.append((k, l))
        print(
            f"[{loop_count}/{loop_length}] "
            f"init={out['init_buckle_tup']} -> desired={out['desired_buckle_tup']} "
            f"loss={out['loss']}"
        )

print("Done.")
print("Failed jobs:", failed_jobs)

In [ ]:
print(f"Finished {len(results)} completed jobs.")
print(f"Number of failed jobs: {len(failed_jobs)}")

successful = {key: val for key, val in results.items() if val['ok']}
best = sorted(successful.items(), key=lambda kv: kv[1]['loss'])[:10]
for (k, l), out in best:
    print(
        f"k={k}, l={l}, init={out['init_buckle_tup']}, desired={out['desired_buckle_tup']}, "
        f"loss={out['loss']:.6g}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(final_loss, cmap=custom_cmap)

tick_labels = [file_funcs.correct_buckle_string(np.array(b).reshape(4, 1)) for b in init_buckles]

ax.set_xticks(range(len(init_buckles)))
ax.set_xticklabels(tick_labels, rotation=90)
ax.set_yticks(range(len(desired_buckles)))
ax.set_yticklabels(tick_labels)

ax.set_xlabel("init buckle")
ax.set_ylabel("desired buckle")
ax.set_title("Final loss across buckle-pair jobs")

cax = ax.inset_axes((1.03, 0, 0.04, 1.0))
cbar = fig.colorbar(im, cax=cax)
cbar.set_label(r"$\|\mathcal{L}\|$")

plt.tight_layout()
plt.show()


In [ ]:
with open('final_loss.npy', 'wb') as f:
    np.save(f, final_loss)


In [ ]:
with open('final_loss.npy', 'rb') as f:
    a = np.load(f)
print(a)
